# Knowledge Layer Demo — Agentic RPG Game Master

This notebook demonstrates the knowledge layer built for the project:

1. Loading initial `GameState` from seed JSON
2. Retrieval helpers that build agent inputs from live state + seed data
3. Running each specialist agent (Lore, NPC, Quest) on real context
4. A full orchestrator turn to show routing + placeholder execution still works

**Owner:** Mazin  
**Depends on:** Christopher's `state_manager.py`, `router.py`, `orchestrator.py` and the agent files in `src/agents/`

## Setup

Make sure the notebook kernel is `Python (agentic_rpg_venv)` and the working directory is the project root so `from src...` imports resolve.

In [11]:
import sys, os
from pathlib import Path

# Walk up until we find the project root
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "src").is_dir():
        project_root = candidate
        break
else:
    raise RuntimeError("Could not find project root containing 'src/'")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.chdir(project_root)

# Load environment variables from .env BEFORE any src imports
from dotenv import load_dotenv
load_dotenv(project_root / ".env")

print("Project root:", project_root)
print("CAP6640_API_KEY loaded:", "yes" if os.environ.get("CAP6640_API_KEY") else "no")

Project root: /Users/mazinbashier/Desktop/Agentic_RPG_Game_Master
CAP6640_API_KEY loaded: yes


## 1. Load initial game state

`state_manager.build_initial_state()` reads `data/world/world_state.json`, `data/npcs/npc_data.json`, and `data/quests/quest_data.json` and returns a fully populated `GameState`.

In [12]:
from src.state_manager import build_initial_state

state = build_initial_state(
    session_id="demo_knowledge_layer",
    player_action="The player steps into the inn at Oakshade.",
)

print("Location:", state.canonical.location)
print("Current scene:", state.canonical.current_scene)
print("NPCs loaded:", list(state.canonical.npc_states.keys()))
print("Quests loaded:", [q.quest_id for q in state.canonical.active_quests])
print("Turn:", state.meta.turn_id, "| Session:", state.meta.session_id)

Location: Oakshade Village
Current scene: The player arrives at Oakshade Village, where rumors, danger, and opportunity are beginning to converge.
NPCs loaded: ['mara_innkeeper', 'corvin_stranger', 'harlan_smith']
Quests loaded: ['missing_scout']
Turn: 1 | Session: demo_knowledge_layer


## 2. Retrieval — build agent inputs

The retrieval layer turns `GameState` + seed JSON into ready-to-run agent inputs. This is the bridge between orchestration and the agents.

In [13]:
from src.retrieval import (
    build_lore_agent_input,
    build_npc_agent_input,
    build_quest_agent_input,
)

lore_input = build_lore_agent_input(state)
print("--- Lore Agent Input ---")
print(lore_input.model_dump_json(indent=2))

--- Lore Agent Input ---
{
  "player_action": "The player steps into the inn at Oakshade.",
  "current_scene": "The player arrives at Oakshade Village, where rumors, danger, and opportunity are beginning to converge.",
  "location": "Oakshade Village",
  "world_summary": "The Shattered Vale is a patchwork of forest-edge villages and ruined holdfasts left behind by a fallen kingdom. Travel between settlements is slow and uncertain, and the Vale's old stone ruins hum with restless things that the living do not fully understand. Oakshade sits at the southern edge of the wood, the last ordered place before the trees grow wild.",
  "relevant_facts": [
    "Several villagers have gone missing near the Emberwood Ruins in the past season.",
    "The Village Council governs Oakshade and is openly worried about the Ashen Band.",
    "Outsiders are tolerated but watched; the innkeeper is the first person most strangers speak to.",
    "The woods between Oakshade and the ruins are considered safe 

In [14]:
npc_input = build_npc_agent_input(state, npc_id="corvin_stranger")
print("--- NPC Agent Input (Corvin) ---")
print(npc_input.model_dump_json(indent=2))

--- NPC Agent Input (Corvin) ---
{
  "player_action": "The player steps into the inn at Oakshade.",
  "npc_name": "Corvin",
  "npc_role": "Hooded traveler at the inn; possible scout or informant with his own agenda",
  "npc_personality": [
    "quiet",
    "guarded",
    "quick to size people up",
    "willing to pay for discreet help"
  ],
  "npc_goals": [
    "Find out what the Ashen Band is pulling out of the Emberwood Ruins",
    "Avoid being identified by the Village Council",
    "Recruit an outsider to do the dangerous part of his investigation"
  ],
  "npc_disposition": "neutral",
  "current_scene": "The player arrives at Oakshade Village, where rumors, danger, and opportunity are beginning to converge.",
  "relationship_summary": "Has been watching the player since they walked in. Will approach if the player seems capable and discreet.",
  "relevant_memory": [
    "Claims to have followed the Ashen Band for weeks before coming to Oakshade.",
    "Has a partial map of the Ember

In [15]:
quest_input = build_quest_agent_input(state)
print("--- Quest Agent Input ---")
print(quest_input.model_dump_json(indent=2))

--- Quest Agent Input ---
{
  "player_action": "The player steps into the inn at Oakshade.",
  "current_scene": "The player arrives at Oakshade Village, where rumors, danger, and opportunity are beginning to converge.",
  "active_quest_summaries": [
    "The Missing Scout: not_started"
  ],
  "completed_objectives": [],
  "recent_player_choices": [],
  "world_context": "The player arrives at Oakshade Village, where rumors, danger, and opportunity are beginning to converge.",
  "npc_context": null
}


## 3. Run the Lore Agent

The player looks around the inn and asks about the village. The Lore Agent grounds its response in Oakshade's seeded facts and rumors.

In [16]:
from src.agents.lore_agent import run_lore_agent
from src.state_manager import start_new_turn

state = start_new_turn(state, "I scan the common room and ask what's been going on in Oakshade lately.")
lore_input = build_lore_agent_input(state)

lore_output = await run_lore_agent(lore_input)
print("--- Lore Agent Output ---")
print(lore_output.model_dump_json(indent=2))

--- Lore Agent Output ---
{
  "summary": "The player scans the common room and inquires about local events. Based on the world context, the innkeeper is the natural first contact for outsiders and would likely be the one to respond or set the tone. The common room would carry the quiet tension of a community under stress — hushed conversations, wary glances at strangers, and the general atmosphere of people who have lost neighbors. The player would pick up on the fear surrounding the Emberwood Ruins and the missing villagers, and may hear guarded references to the Ashen Band. No major lore changes result from this action; it is a standard social information-gather.",
  "environmental_details": [
    "The common room is dimly lit, with a low fire and a few scattered patrons who fall quiet or shift their gaze when a stranger speaks up.",
    "The innkeeper stands behind a worn wooden bar, polishing a cup — the default social anchor for outsiders in Oakshade.",
    "One corner table holds

## 4. Run the NPC Agent

The player approaches Corvin, the hooded stranger. The NPC Agent should respond in Corvin's voice — guarded, watchful, and hinting at his own agenda.

In [17]:
from src.agents.npc_agent import run_npc_agent

state = start_new_turn(state, "I walk over to the hooded man in the corner and sit down across from him.")
npc_input = build_npc_agent_input(state, npc_id="corvin_stranger")

npc_output = await run_npc_agent(npc_input)
print("--- NPC Agent Output (Corvin) ---")
print(npc_output.model_dump_json(indent=2))

--- NPC Agent Output (Corvin) ---
{
  "speech": "Corvin doesn't look up right away. He keeps his eyes on the cup in front of him, turning it slowly with two fingers. A beat passes. Then another. When he finally raises his gaze, it's measured — sizing you up the way a man does when his life might depend on the answer. \"You've got a traveler's look about you,\" he says, voice low and even, just above the noise of the room. \"Not from Oakshade. Not looking to stay, either, I'd wager.\" He leans back slightly, putting an inch more distance between you without making it obvious. His hood stays up. \"Most people who sit across from a stranger in a quiet corner are either lost... or looking for something. Which are you?\"",
  "emotional_tone": "Guarded, calculating, cautiously curious — not hostile, but every word is measured and deliberate. He's testing before he trusts.",
  "intent": "Corvin is assessing whether the player is a threat, a fool, or a potential asset. He won't reveal anything

### NPC comparison — the same action at Mara

Running the same turn against a different NPC should produce a clearly different voice. This is a quick sanity check that personality/role/goals actually flow through.

In [18]:
state = start_new_turn(state, "I lean on the bar and ask the innkeeper what's been happening around here.")
npc_input_mara = build_npc_agent_input(state, npc_id="mara_innkeeper")

npc_output_mara = await run_npc_agent(npc_input_mara)
print("--- NPC Agent Output (Mara) ---")
print(npc_output_mara.model_dump_json(indent=2))

--- NPC Agent Output (Mara) ---
{
  "speech": "*pauses mid-wipe of the bar counter, sizing you up for a beat before setting the cloth down*\n\n\"Happening around here.\" *she repeats it flatly, like she's tasting the question.* \"Well. Harvest season's been dry, road from the east got washed out two weeks back, and folks are drinking more than usual — make of that what you will.\" *She slides a cup of something warm toward you without being asked, then leans her own arms on the bar, mirroring your posture just slightly.*\n\n\"You asking because you're passing through, or because someone sent you?\" *Her tone isn't hostile — it's measured. Professional. The kind of calm that comes from years of reading rooms.* \"I've had three strangers through this week already. That's two more than usual. So you'll forgive me if I'm a little… curious about what's pulling people toward Oakshade all of a sudden.\"\n\n*She doesn't answer her own question. She waits.*",
  "emotional_tone": "Guarded, measu

## 5. Run the Quest Agent

The player accepts Mara's request to look into the missing scout. The Quest Agent should interpret this as advancing the `speak_to_mara` objective.

In [19]:
from src.agents.quest_agent import run_quest_agent

state = start_new_turn(state, "I tell Mara I'll look into what happened to the missing scout.")
quest_input = build_quest_agent_input(state)

quest_output = await run_quest_agent(quest_input)
print("--- Quest Agent Output ---")
print(quest_output.model_dump_json(indent=2))

--- Quest Agent Output ---
{
  "progress_interpretation": "The player's action of accepting Mara's request to investigate the missing scout directly advances \"The Missing Scout\" quest from not_started to in_progress. This is a clear quest acceptance moment — the player has acknowledged the quest hook and committed to pursuing it. No prior objectives were completed; this action serves as the formal uptake of the quest.",
  "objective_status_updates": [
    "The Missing Scout — Quest status: not_started → in_progress",
    "New objective unlocked: Gather information about the missing scout's last known whereabouts or route"
  ],
  "new_branches_or_hooks": [
    "Hook: Mara may provide a name, description, or last known location of the scout, opening a lead to follow outside Oakshade Village",
    "Hook: Other villagers in Oakshade may have seen or heard something relevant — potential for additional witness interviews",
    "Branch: If the scout's disappearance is linked to a larger thr

## 6. Full orchestrator turn (routing + placeholder execution)

Finally, confirm that Christopher's orchestrator still runs end-to-end with the new seed data. The agents aren't wired in yet — nodes execute as placeholders — but routing, state transitions, and event logging all flow through.

In [20]:
from src.orchestrator import run_turn

result = run_turn(state, player_action="I ask Mara who else I should talk to in the village.")

print("Selected route:", result.selected_route)
print("Next nodes:   ", result.next_nodes)
print("Execution plan:", result.execution_plan)
print("Aborted:      ", result.aborted)
print("Reason:       ", result.reason)
print()
print("Execution results:")
for r in result.execution_results:
    print(f"  - {r.node_name}: {r.status} ({r.attempts} attempt(s)) — {r.summary}")

Selected route: dialogue
Next nodes:    ['npc_agent']
Execution plan: ['npc_agent']
Aborted:       False
Reason:        Player action appears to be directed at a character or conversation.

Execution results:
  - npc_agent: completed (1 attempt(s)) — Placeholder execution completed for npc_agent.


In [21]:
print("--- Turn events ---")
for e in result.state.turn.current_turn_events:
    print(f"[{e.event_type}] ({e.source_node}) {e.summary}")

--- Turn events ---
[route_selected] (orchestrator) Route 'dialogue' selected with nodes ['npc_agent'].
[route_reason] (router) Player action appears to be directed at a character or conversation.
[execution_plan_built] (orchestrator) Execution plan created: ['npc_agent']
[node_executed] (orchestrator) npc_agent executed with placeholder behavior.


## What this proves

- Seed JSON loads cleanly into a validated `GameState`.
- Retrieval packages that state into the correct input shape for all three agents.
- Each agent runs against real seeded context and returns structured output.
- Different NPCs produce clearly different voices from the same kind of prompt.
- Christopher's orchestrator still routes and executes turns correctly with the new data.

## What's still placeholder

- The orchestrator executes nodes as placeholders rather than calling the real agents.
- Wiring `run_lore_agent`, `run_npc_agent`, and `run_quest_agent` into `execute_planned_nodes` is the next orchestration-side step — ideally coordinated with Christopher.